# Importing the libraries

In [26]:
import base64
import requests
import pandas as pd
import re
import os
import time
from openai import OpenAI

# Setting up the API

In [27]:
# Initialize OpenAI client
client = OpenAI()

# Loading the data

In [28]:
csv_file_path = 'Sample to rate/CSV/AllResults.csv'

# Load the CSV file into a DataFrame
df = pd.read_csv(csv_file_path, sep=';')

# Defining the helper functions

## Helper function to encode an image in base64 format

In [29]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

## Helper function to call the API with images only

In [30]:
def get_response_with_images(question, image_paths):
    encoded_images = [encode_image(path) for path in image_paths]
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": question,
                }
            ] + [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{encoded_image}"
                    }
                } for encoded_image in encoded_images
            ]
        }
    ]
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=1000
    )
    return response.choices[0].message.content


# Defining the unchanged components of the prompts

## Explanations

In [31]:
# Define explanations
explanations = "followed by your step-by-step reasoning"

## Image Example 

In [32]:
image_directory_2 = 'Examples/Images'

## Input Example

In [33]:
input_example_file_path = 'Examples/Text/input_text.txt'

# Read the contents of the report text file
with open(input_example_file_path, 'r', encoding='utf-8') as file:
    input_example = file.read()

## Output Example

In [34]:
output_example1_file_path = 'Examples/Text/Example1.txt'
output_example2_file_path = 'Examples/Text/Example2.txt'
output_example3_file_path = 'Examples/Text/Example3.txt'
output_example4_file_path = 'Examples/Text/Example4.txt'


# Read the contents of the report text file
with open(output_example1_file_path, 'r', encoding='utf-8') as file:
    output_example1 = file.read()
    
# Read the contents of the report text file
with open(output_example2_file_path, 'r', encoding='utf-8') as file:
    output_example2 = file.read()  
    
# Read the contents of the report text file
with open(output_example3_file_path, 'r', encoding='utf-8') as file:
    output_example3 = file.read()
    
# Read the contents of the report text file
with open(output_example4_file_path, 'r', encoding='utf-8') as file:
    output_example4 = file.read()

# Faithfulness

In [38]:
import os

# Assuming df is your DataFrame
# Loop through the first row in the DataFrame
for index, row in df.iterrows():
    # Extract values from the specified columns
    input_text = row['Input']
    report = row['Output']
    case_study = row['Case study']
    question = f"""Your task is to rate a report based on its faithfulness with respect to an input context and input plots. Faithfulness is on a scale from 1 (worst) to 5 (perfect). Your answer must state the number you give for faithfulness {explanations}.
    Consider the following examples. For example, the input context is: {input_example} and the input plot is given to you as an attachment and tagged with the label "Image used for the example".
    An example of a report is: {output_example1}. This example scores 3 on faithfulness, because it states several values incorrectly and it does not mention the final values shown in the plot. A second example of a report is: {output_example2}. This second example scores 3 on faithfulness, because it states several values incorrectly and it does not mention the final values shown in the plot. A third example of a report is: {output_example3}. This third example scores 1 on faithfulness, because there it lacks context, it omits the overall evolution, it states several values incorrectly, and it misses intermediate trends from the plot. A last example is: {output_example4}. This last report scores 1 on faithfulness, because there it lacks context, it omits the overall evolution, it states several values incorrectly, and it misses intermediate trends from the plot.
    In your case, the input context is: {input_text} The accompanying input plots are given to you as an attachment. 
    The report that you must rate for faithfulness is {report}"""
    
    image_directory_1 = f'Sample to rate/Images/{case_study}'
    
    # Check if the directory exists
    if os.path.exists(image_directory_1) and os.path.isdir(image_directory_1):
        # List all image files in the directory
        image_paths = [os.path.join(image_directory_1, f) for f in os.listdir(image_directory_1) if f.endswith('.png')]
    else:
        image_paths = []
        
    # Add the additional image path from the second directory
    additional_image_path = os.path.join('Examples/Images', 'Picture Example.png')
    image_paths.append(additional_image_path)
    
    # Get the response from the API
    api_response = get_response_with_images(question, image_paths)
    
    # Update the DataFrame with the API response and additional columns
    df.at[index, 'LLM Faithfulness Raw Response'] = api_response
    df.at[index, 'Explanations for Evaluation'] = 'Yes'
    df.at[index, 'Number of examples for evaluation'] = 4
    
    # Save the DataFrame to a CSV file after processing each row
    df.to_csv('updated_dataframe.csv', index=False)
    
    # Print or use the variables as needed for debugging
    print(f"Input: {input_text}")
    print(f"Output: {report}")
    print(f"Images: {image_paths}")
    print(f"API Response: {api_response}")

# Save or further process the DataFrame as needed
# df.to_csv('updated_dataframe.csv', index=False)


Input: You are an expert in Consumer Behavior without any statistics background.



Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, from 0.0 to 1.0. We set it to 0.15.
  - network-parameter, from

Input: You are an expert in Consumer Behavior without any statistics background.



Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, from 0.0 to 1.0. We set it to 0.15.
  - network-parameter, from

Input: You are an expert in Consumer Behavior without any statistics background.



Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, from 0.0 to 1.0. We set it to 0.15.
  - network-parameter, from

Input: You are an expert in Consumer Behavior without any statistics background.



Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, from 0.0 to 1.0. We set it to 0.15.
  - network-parameter, from

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, from 0.0 to 1.0. We set it to 0.15.
  - network-parameter, from 2.0 to 10.0. We set it to 8.
  - number-of-agents. We set it to 1000.
  - ha

...

Input: You are an expert in megaherbivore extinction without any statistics background.



Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-foo

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-food, from 0.0 to 10.0. We set it to 1.5.
  - animals-starvation-threshold, from 0.0 to

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-food, from 0.0 to 10.0. We set it to 1.5.
  - animals-starvation-threshold, from 0.0 to

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-food, from 0.0 to 10.0. We set it to 1.5.
  - animals-starvation-threshold, from 0.0 to

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-food, from 0.0 to 10.0. We set it to 1.5.
  - animals-starvation-threshold, from 0.0 to

# Coverage

In [40]:
import os

# Assuming df is your DataFrame
# Loop through the rows in the DataFrame
for index, row in df.iterrows():
    # Check if the 'LLM Coverage Raw Response' column is empty
    if pd.isna(row['LLM Coverage Raw Response']) or row['LLM Coverage Raw Response'] == "":
        # Extract values from the specified columns
        input_text = row['Input']
        report = row['Output']
        case_study = row['Case study']
        question = f"""Your task is to rate a report based on its coverage with respect to an input context and input plots. Coverage is on a scale from 1 (worst) to 5 (perfect). Your answer must state the number you give for coverage {explanations}.
        Consider the following examples. For example, the input context is: {input_example} and the input plot is given to you as an attachment and tagged with the label "Image used for the example".
        An example of a report is: {output_example1}. This example scores 5 on coverage, because it only provided information that was in the input (no hallucinations). Another example of a report is: {output_example2}. This example scores 3 on coverage: it lost two points because it gave two pieces of information that were not in the input (car brand, model commissioner). A third report is : {output_example3}. This example scores 5 on coverage, because it only provided information that was in the input (no hallucinations). A last example is : {output_example4}. This last example scores 3 on coverage: it lost two points because it gave two pieces of information that were not in the input (car brand, model commissioner).
        In your case, the input context is: {input_text} The accompanying input plots are given to you as an attachment. 
        The report that you must rate for coverage is {report}"""
        
        image_directory_1 = f'Sample to rate/Images/{case_study}'
        
        # Check if the directory exists
        if os.path.exists(image_directory_1) and os.path.isdir(image_directory_1):
            # List all image files in the directory
            image_paths = [os.path.join(image_directory_1, f) for f in os.listdir(image_directory_1) if f.endswith('.png')]
        else:
            image_paths = []
            
        # Add the additional image path from the second directory
        additional_image_path = os.path.join('Examples/Images', 'Picture Example.png')
        image_paths.append(additional_image_path)
        
        # Get the response from the API
        api_response = get_response_with_images(question, image_paths)
        
        # Update the DataFrame with the API response
        df.at[index, 'LLM Coverage Raw Response'] = api_response
        
        # Save the DataFrame to a CSV file after processing each row
        df.to_csv('updated_dataframe.csv', index=False)
        
        # Print or use the variables as needed for debugging
        print(f"Input: {input_text}")
        print(f"Output: {report}")
        print(f"Images: {image_paths}")
        print(f"API Response: {api_response}")
    else:
        print(f"Skipping row {index} as 'LLM Coverage Raw Response' is not empty.")

# Save or further process the DataFrame as needed
# df.to_csv('updated_dataframe.csv', index=False)


Skipping row 0 as 'LLM Coverage Raw Response' is not empty.
Input: You are an expert in Consumer Behavior without any statistics background.



Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, fro

Input: You are an expert in Consumer Behavior without any statistics background.



Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, from 0.0 to 1.0. We set it to 0.15.
  - network-parameter, from

Input: You are an expert in Consumer Behavior without any statistics background.



Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, from 0.0 to 1.0. We set it to 0.15.
  - network-parameter, from

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, from 0.0 to 1.0. We set it to 0.15.
  - network-parameter, from 2.0 to 10.0. We set it to 8.
  - number-of-agents. We set it to 1000.
  - ha

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 16 parameters:
  - memory-lifetime, from 1.0 to 10.0. We set it to 5.
  - alt-health-mean-initial, from 1.0 to 3.0. We set it to 1.69.
  - habit-threshold, from 1.0 to 10.0. We set it to 2.
  - p-interact, from 0.0 to 1.0. We set it to 0.3.
  - social-susceptibility, from 0.0 to 1.0. We set it to 0.2.
  - incumbent-initial-habit, from 0.0 to 10.0. We set it to 10.
  - alt-env-mean-initial, from 0.0 to 3.0. We set it to 1.12.
  - social-blindness, from 0.0 to 1.0. We set it to 0.51.
  - justification, from 0.0 to 1.0. We set it to 0.61.
  - cognitive-dissonance-threshold, from 0.0 to 1.0. We set it to 0.15.
  - network-parameter, from 2.0 to 10.0. We set it to 8.
  - number-of-agents. We set it to 1000.
  - ha

...

Input: You are an expert in megaherbivore extinction without any statistics background.



Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-foo

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-food, from 0.0 to 10.0. We set it to 1.5.
  - animals-starvation-threshold, from 0.0 to

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-food, from 0.0 to 10.0. We set it to 1.5.
  - animals-starvation-threshold, from 0.0 to

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-food, from 0.0 to 10.0. We set it to 1.5.
  - animals-starvation-threshold, from 0.0 to

Input: Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters. For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion. We have 29 parameters:
  - init-foragers, from 0.0 to 100.0. We set it to 20.
  - processing-cost-males, from 0.0 to 20.0. We set it to 6.
  - food-value-males, from 5.0 to 100.0. We set it to 30.
  - maximum-lifespan, from 0.0 to 12000.0. We set it to 10220.
  - processing-cost-females, from 0.0 to 20.0. We set it to 3.
  - food-value-females, from 5.0 to 100.0. We set it to 27.
  - init-prey, from 0.0 to 1000.0. We set it to 250.
  - internal-mortality, from 0.0 to 0.2. We set it to 0.025.
  - age-of-senescence, from 0.0 to 10000.0. We set it to 7300.
  - ticks-per-year, from 0.0 to 720.0. We set it to 365.
  - animals-gain-from-food, from 0.0 to 10.0. We set it to 1.5.
  - animals-starvation-threshold, from 0.0 to